# Delivery Time Regression


In [1]:
import sys
from pathlib import Path

# Find the project root by walking up until src/olist is visible, so the
# notebook runs from any working directory.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "olist").exists())
sys.path.insert(0, str(ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Ridge

from olist import config as cfg, io_utils as io, modeling as ml, plots
from olist.features import PURCHASE_TIME_FEATURES

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 220)
cfg.set_seed()

PROJECT = "02_delivery_time_regression"
TARGET = "delivery_days"
paths = ml.get_paths(PROJECT)

df = io.load_ml("order_model_base")

## 1. Feature contract


In [2]:
FEATURES = [c for c in PURCHASE_TIME_FEATURES if c in df.columns]
audit = ml.leakage_audit(FEATURES, TARGET)

In [3]:
data = ml.chronological_split(df, FEATURES, TARGET)

## 2. Datasets


In [4]:
y_test = data.y_test
X_test = data.X_test

In [5]:
promise_pred = X_test["estimated_delivery_days"].to_numpy(dtype=float)
promise_metrics = ml.regression_metrics(y_test, promise_pred)

In [6]:
train_mean = float(data.y_train.mean())
mean_metrics = ml.regression_metrics(y_test, np.full(len(y_test), train_mean))

In [7]:
# Distance-only OLS, fitted on train only.
dist_train = data.X_train["distance_km"].to_numpy(dtype=float)
mask = ~np.isnan(dist_train)
slope, intercept = np.polyfit(dist_train[mask], data.y_train.to_numpy(dtype=float)[mask], 1)
dist_test = np.nan_to_num(X_test["distance_km"].to_numpy(dtype=float), nan=float(np.nanmean(dist_train)))
distance_pred = intercept + slope * dist_test
distance_metrics = ml.regression_metrics(y_test, distance_pred)

print(f"distance rule fitted on train: delivery_days = {intercept:.2f} + {slope:.5f} * distance_km")
print(f"  i.e. +1 day per {1 / slope:,.0f} km\n")

distance rule fitted on train: delivery_days = 9.62 + 0.00662 * distance_km
  i.e. +1 day per 151 km



In [8]:
baselines = pd.DataFrame({
    "predict the train mean": mean_metrics,
    "distance-only linear rule": distance_metrics,
    "INCUMBENT: Olist promise date": promise_metrics,
}).T
baselines.round(3)

,mae,rmse,r2,mape_pct,median_abs_err
predict the train mean,6.662,7.587,-0.995,141.850,6.675
distance-only linear rule,6.097,7.062,-0.729,114.692,5.945
INCUMBENT: Olist promise date,9.722,12.091,-4.067,153.038,8.224


In [9]:
# How does the incumbent promise actually behave? Accurate is not the same as safe.
promise_error = promise_pred - y_test.to_numpy(dtype=float)
print(f"incumbent promise MAE : {promise_metrics['mae']:.2f} days")
print(f"mean signed error     : {promise_error.mean():+.2f} days (positive = promised later than reality)")
print(f"orders arriving late   : {(promise_error < 0).mean():.1%}")

incumbent promise MAE : 9.72 days
mean signed error     : +9.07 days (positive = promised later than reality)
orders arriving late   : 7.5%


## 3. Model candidates


In [10]:
candidates = {
    **ml.baseline_regressors(),
    "ridge": Ridge(alpha=10.0, random_state=cfg.RANDOM_STATE),
    "random_forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=20,
                                           random_state=cfg.RANDOM_STATE, n_jobs=cfg.N_JOBS),
    "hist_gbm": HistGradientBoostingRegressor(max_iter=350, learning_rate=0.06,
                                              max_leaf_nodes=31, l2_regularization=1.0,
                                              random_state=cfg.RANDOM_STATE),
}
comparison, fitted = ml.compare_candidates(candidates, data, task="regression")
comparison.round(3)

,model,mae,rmse,r2,mape_pct,median_abs_err
0,hist_gbm,4.389,6.372,0.270,56.956,3.280
1,random_forest,4.510,6.530,0.233,58.552,3.375
2,baseline_median,5.562,7.516,-0.016,93.718,4.811
3,baseline_mean,6.593,8.159,-0.197,122.421,6.429
4,ridge,6.642,8.290,-0.236,94.915,5.838


In [11]:
BEST_NAME = comparison.iloc[0]["model"]
best_pipe = fitted[BEST_NAME]
print("selected on validation MAE:", BEST_NAME)

pred = best_pipe.predict(ml.to_sklearn_frame(X_test))
model_metrics = ml.regression_metrics(y_test, pred)

scoreboard = pd.DataFrame({
    "baseline: train mean": mean_metrics,
    "baseline: distance rule": distance_metrics,
    "INCUMBENT: Olist promise": promise_metrics,
    f"model: {BEST_NAME}": model_metrics,
}).T
print(scoreboard.round(3).to_string())
print()
improvement = 1 - model_metrics["mae"] / promise_metrics["mae"]
print(f"MAE vs the incumbent promise: {improvement:+.1%}")
print(f"  incumbent {promise_metrics['mae']:.2f} days -> model {model_metrics['mae']:.2f} days")
print(f"  R2 {model_metrics['r2']:.3f} (variance explained on unseen future orders)")

selected on validation MAE: hist_gbm
                            mae    rmse     r2  mape_pct  median_abs_err
baseline: train mean      6.662   7.587 -0.995   141.850           6.675
baseline: distance rule   6.097   7.062 -0.729   114.692           5.945
INCUMBENT: Olist promise  9.722  12.091 -4.067   153.038           8.224
model: hist_gbm           4.091   5.633 -0.100    61.995           3.064

MAE vs the incumbent promise: +57.9%
  incumbent 9.72 days -> model 4.09 days
  R2 -0.100 (variance explained on unseen future orders)


## 4. Diagnostics


In [12]:
fig = plots.plot_regression_diagnostics(
    y_test, pred, unit="days", by=X_test["distance_bucket"],
)

In [13]:
plots.stamp_and_save(
    fig, paths.figure("diagnostics.png"),
    f"INSIGHT: MAE {model_metrics['mae']:.2f} days, median absolute error "
    f"{model_metrics['median_abs_err']:.2f} days, R2 {model_metrics['r2']:.3f} on out-of-time orders. "
    f"The residual spread widens with distance, so a single MAE understates the uncertainty on long "
    f"hauls - which is precisely why section 7 promises a quantile rather than the mean."
)
plt.show()

In [14]:
diag = X_test.copy()
diag["actual"] = y_test.to_numpy()
diag["predicted"] = pred
diag["error"] = diag["predicted"] - diag["actual"]

In [15]:
by_band = diag.groupby("distance_bucket", observed=True).agg(
    orders=("actual", "size"), mean_actual=("actual", "mean"), mean_pred=("predicted", "mean"),
    mae=("error", lambda s: s.abs().mean()), bias=("error", "mean"),
)
by_band = by_band[by_band["orders"] >= 25].sort_values("mae")

In [16]:
print("error by distance band (min 25 orders):")
print(by_band.round(2).to_string())
print()

error by distance band (min 25 orders):
                 orders  mean_actual  mean_pred   mae  bias
distance_bucket                                            
<50km              1946         5.09       5.15  2.44  0.06
50-200km           1823         5.83       6.17  2.47  0.34
200-500km          3758         7.87      10.02  3.54  2.15
500-1000km         3037         9.56      12.26  4.34  2.70
1000km+            1892        12.99      19.71  8.02  6.71



In [17]:
worst = by_band.index[-1]
print(f"Worst band: {worst}, MAE {by_band.loc[worst, 'mae']:.2f} days, "
      f"bias {by_band.loc[worst, 'bias']:+.2f} days.")


Worst band: 1000km+, MAE 8.02 days, bias +6.71 days.


In [18]:
# Where the model is confidently wrong. These are the rows to look at by hand.
worst_misses = diag.reindex(diag["error"].abs().sort_values(ascending=False).index).head(10)
print("ten largest misses:")
print(worst_misses[["actual", "predicted", "error", "distance_km", "estimated_delivery_days",
                    "seller_prior_late_rate", "order_weight_g"]].round(1).to_string())
print()
print(f"the top 1% of absolute errors account for "
      f"{diag['error'].abs().nlargest(max(1, len(diag) // 100)).sum() / diag['error'].abs().sum():.1%} "
      f"of total absolute error => a small tail of stuck shipments dominates, and no")


ten largest misses:
       actual  predicted  error  distance_km  estimated_delivery_days  seller_prior_late_rate  order_weight_g
88302    68.0        4.4  -63.6         28.3                      7.4                     0.1          2450.0
88829    68.9       11.7  -57.2        844.0                     21.9                     0.0           450.0
87163    77.7       26.5  -51.2       2274.7                     27.1                     0.1           900.0
92322    63.6       14.0  -49.6       1342.5                     14.6                     0.1          2500.0
90455    66.7       18.5  -48.2       2037.0                     25.6                     0.1          4650.0
90098    59.0       11.8  -47.2        539.1                     26.4                     0.2           233.0
87094    59.3       13.4  -46.0        748.6                     27.3                     0.1           350.0
91261    54.2       10.3  -43.9        540.7                     15.6                     0.1       

## 5. Feature importance


In [19]:
importance = ml.importance_table(
    best_pipe, X_test, y_test, top_k=15, n_repeats=5, scoring="neg_mean_absolute_error"
)
print(importance.round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 6))
top = importance.head(10).iloc[::-1]
ax.barh(top["feature"], top["importance"], xerr=top["std"], color="#2a9d8f")
ax.set(xlabel="Increase in MAE when the column is shuffled (days)",
       title="What the delivery-time model relies on")
ax.grid(axis="x", alpha=0.3)
plots.stamp_and_save(
    fig, paths.figure("importance.png"),
    f"INSIGHT: '{importance.iloc[0]['feature']}' dominates, worth "
    f"{importance.iloc[0]['importance']:.2f} days of MAE on its own, followed by "
    f"'{importance.iloc[1]['feature']}'. Physical and geographic features carry the signal; basket "
    f"composition contributes little, which matches the intuition that transit time is a network "
    f"property, not a shopping-cart property."
)
plt.show()

                feature  importance   std
         Customer_State      -0.274 0.010
    shipping_limit_days       0.098 0.006
estimated_delivery_days       0.090 0.005
       order_volume_cm3       0.057 0.002
         avg_item_price       0.056 0.002
       distance_km_mean      -0.054 0.006
           Seller_State       0.041 0.002
  route_prior_late_rate      -0.040 0.005
         freight_per_kg       0.028 0.001
         max_item_price       0.022 0.001
  dominant_payment_type       0.021 0.002
     route_prior_orders       0.020 0.002
          main_category       0.018 0.002
   purchase_weekday_num       0.018 0.004
        same_state_flag      -0.017 0.010


## 6. Validation


In [20]:
valid_pred = best_pipe.predict(ml.to_sklearn_frame(data.X_valid))
valid_residual = data.y_valid.to_numpy(dtype=float) - valid_pred  

policy_rows = []
for service_level in [50, 75, 90, 95, 99]:
    buffer_days = float(np.percentile(valid_residual, service_level))
    promise = pred + buffer_days
    late = promise < y_test.to_numpy(dtype=float)
    policy_rows.append({
        "service_level_pct": service_level,
        "buffer_days": round(buffer_days, 2),
        "mean_promise_days": round(float(promise.mean()), 2),
        "achieved_on_time_pct": round(float((~late).mean() * 100), 1),
        "late_rate_pct": round(float(late.mean() * 100), 1),
        "mean_padding_days": round(float((promise - y_test.to_numpy(dtype=float)).mean()), 2),
    })

In [21]:
policy = pd.DataFrame(policy_rows)

incumbent_row = {
    "service_level_pct": "incumbent",
    "buffer_days": np.nan,
    "mean_promise_days": round(float(promise_pred.mean()), 2),
    "achieved_on_time_pct": round(float((promise_pred >= y_test.to_numpy(dtype=float)).mean() * 100), 1),
    "late_rate_pct": round(float((promise_pred < y_test.to_numpy(dtype=float)).mean() * 100), 1),
    "mean_padding_days": round(float(promise_error.mean()), 2),
}
pd.concat([pd.DataFrame([incumbent_row]), policy], ignore_index=True)

,service_level_pct,buffer_days,mean_promise_days,achieved_on_time_pct,late_rate_pct,mean_padding_days
0,incumbent,NaN,17.40,92.5,7.5,9.07
1,50,-1.91,8.80,56.0,44.0,0.47
2,75,0.89,11.60,82.4,17.6,3.26
3,90,5.81,16.53,95.9,4.1,8.19
4,95,9.32,20.03,98.2,1.8,11.69
5,99,18.44,29.15,99.5,0.5,20.82


In [22]:
# Find the model policy that matches the incumbent's late rate, then compare padding.
target_late = incumbent_row["late_rate_pct"]
matched = policy.iloc[(policy["late_rate_pct"] - target_late).abs().argsort()].iloc[0]
padding_saved = incumbent_row["mean_padding_days"] - matched["mean_padding_days"]

In [23]:
print(f"incumbent promise : {incumbent_row['mean_promise_days']:.1f} days average, "
      f"{incumbent_row['late_rate_pct']:.1f}% late, {incumbent_row['mean_padding_days']:.1f} days of padding")
print(f"matched model policy (p{matched['service_level_pct']:.0f} buffer "
      f"{matched['buffer_days']:.1f} days): {matched['mean_promise_days']:.1f} days average, "
      f"{matched['late_rate_pct']:.1f}% late, {matched['mean_padding_days']:.1f} days of padding")
print()
print(f"=> at the SAME late rate the model promises {padding_saved:+.1f} days "
      f"{'earlier' if padding_saved > 0 else 'later'} on average.")

incumbent promise : 17.4 days average, 7.5% late, 9.1 days of padding
matched model policy (p90 buffer 5.8 days): 16.5 days average, 4.1% late, 8.2 days of padding

=> at the SAME late rate the model promises +0.9 days earlier on average.


## 7. Save artifacts


In [24]:
saved = ml.save_artifacts(
    best_pipe, paths, target=TARGET, features=FEATURES,
    metrics=model_metrics,
    baseline_metrics={f"incumbent_promise_{k}": v for k, v in promise_metrics.items()},
    policy={
        "type": "quantile_promise",
        "buffer_source": "validation residual percentile",
        "recommended_service_level_pct": float(matched["service_level_pct"]),
        "buffer_days": float(matched["buffer_days"]),
        "achieved_on_time_pct": float(matched["achieved_on_time_pct"]),
        "padding_days_saved_vs_incumbent": float(padding_saved),
    },
    model_name=BEST_NAME,
    notes=("Baseline is the incumbent Olist promise date, not a dummy. The deliverable is a "
           "quantile promise policy; the point prediction alone would be late ~50% of the time."),
)
for kind, path in saved.items():
    print(f"{kind:<10} -> {path.relative_to(ROOT)}")

model      -> models\02_delivery_time_regression\best_model.joblib
metadata   -> models\02_delivery_time_regression\model_metadata.json
